# ViT-Ctr Phase 3: Bootstrap不确定性量化（AutoDL版）
在完成基础训练后运行。需要 `/root/autodl-tmp/checkpoints/best_model.pth`。

**策略 (D-12/D-13):** 冻结SimpViT backbone，200次有放回重采样训练集，每次微调fc输出头5个epoch。

**注意:** 每次迭代约6分钟（RTX 3090），共200次 ≈ 20小时。推荐在持久化会话（tmux/screen）中运行。
AutoDL数据已在本地 `/root/autodl-tmp/`，无需挂载Drive。

In [ ]:
import sys, os
CODE_DIR = '/root/autodl-tmp/ViT-Ctr'
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))
print(f"Code directory: {CODE_DIR}")

In [ ]:
import os, json, torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOCAL_DATA_DIR = '/root/autodl-tmp/data'
CHECKPOINT_DIR = '/root/autodl-tmp/checkpoints'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
H5_PATHS = [os.path.join(LOCAL_DATA_DIR, f) for f in ['dithioester.h5', 'trithiocarbonate.h5', 'xanthate.h5', 'dithiocarbamate.h5']]
BASE_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
BOOTSTRAP_HEADS_PATH = os.path.join(CHECKPOINT_DIR, 'bootstrap_heads.pth')
CALIBRATION_PATH = os.path.join(CHECKPOINT_DIR, 'calibration.json')
N_BOOTSTRAP = 200
N_EPOCHS_PER_HEAD = 5
BOOTSTRAP_LR = 1e-3
print(f'Device: {DEVICE}, N_BOOTSTRAP={N_BOOTSTRAP}')

In [ ]:
from model import SimpViT
model = SimpViT(num_outputs=3).to(DEVICE)
ckpt = torch.load(BASE_MODEL_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded base model (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.6f})")

In [ ]:
from dataset import CombinedHDF5Dataset
from utils.split import build_stratified_indices
from torch.utils.data import DataLoader
train_idx, val_idx, test_idx = build_stratified_indices(H5_PATHS, seed=42)
train_ds = CombinedHDF5Dataset(H5_PATHS, train_idx)
val_ds = CombinedHDF5Dataset(H5_PATHS, val_idx)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
print('[INFO] 运行3次调试迭代以检查收敛...')
from bootstrap import run_bootstrap
debug_ckpt = run_bootstrap(model, train_loader, n_bootstrap=3, n_epochs=N_EPOCHS_PER_HEAD, lr=BOOTSTRAP_LR, device=DEVICE, seed=999)
print(f'调试完成，得到 {len(debug_ckpt["heads"])} 个头')
print('如果loss在每次迭代最后几个epoch明显下降，增加N_EPOCHS_PER_HEAD')

In [ ]:
bootstrap_ckpt = run_bootstrap(model, train_loader, n_bootstrap=N_BOOTSTRAP, n_epochs=N_EPOCHS_PER_HEAD, lr=BOOTSTRAP_LR, device=DEVICE, seed=42)
torch.save(bootstrap_ckpt, BOOTSTRAP_HEADS_PATH)
print(f'Bootstrap完成，保存到 {BOOTSTRAP_HEADS_PATH}')

In [ ]:
import numpy as np
from bootstrap import compute_jci, compute_coverage, calibrate_coverage
from evaluate import run_inference
y_true_val, _, _ = run_inference(model, val_loader, DEVICE)
val_preds_all = []
model.load_state_dict(bootstrap_ckpt['base_model_state_dict'])
model.eval()
import copy
base_state = bootstrap_ckpt['base_model_state_dict']
with torch.no_grad():
    for head_state in bootstrap_ckpt['heads']:
        full_state = copy.deepcopy(base_state)
        full_state.update(head_state)
        model.load_state_dict(full_state)
        batch_preds = []
        for fp, _ in val_loader:
            batch_preds.append(model(fp.to(DEVICE)).cpu().numpy())
        val_preds_all.append(np.vstack(batch_preds))
val_preds_all = np.stack(val_preds_all)
val_pred_mean = val_preds_all.mean(axis=0)
val_pred_std = val_preds_all.std(axis=0)
from scipy.stats import f as fdist
p = 3; n = N_BOOTSTRAP; dfd = n - p
f_val = fdist.ppf(0.95, dfn=p, dfd=dfd)
val_pred_half = np.sqrt(val_pred_std**2 * p * f_val / dfd)
raw_coverage = compute_coverage(y_true_val, val_pred_mean, val_pred_half)
print(f'Raw coverage: {raw_coverage}')
cal_factors = calibrate_coverage(y_true_val, val_pred_mean, val_pred_half, target=0.95)
calibrated_half = val_pred_half * np.array(cal_factors)
final_coverage = compute_coverage(y_true_val, val_pred_mean, calibrated_half)
print(f'Coverage after calibration: {final_coverage}')
print(f'Calibration factors: {cal_factors}')

In [ ]:
cal_result = {'cal_factors': cal_factors, 'empirical_coverage_before': raw_coverage.tolist(), 'empirical_coverage_after': final_coverage.tolist(), 'n_val_samples': len(y_true_val), 'target_coverage': 0.95, 'debug_mode': bootstrap_ckpt['debug_mode']}
with open(CALIBRATION_PATH, 'w') as f:
    json.dump(cal_result, f, indent=2)
print(f'校准结果保存到 {CALIBRATION_PATH}')

In [ ]:
from bootstrap import predict_with_uncertainty
bs_ckpt = torch.load(BOOTSTRAP_HEADS_PATH, map_location='cpu', weights_only=False)
model_cpu = SimpViT(num_outputs=3)
test_ds_demo = CombinedHDF5Dataset(H5_PATHS, test_idx[:1])
fp_demo, lbl_demo = test_ds_demo[0]
mean_pred, half_width = predict_with_uncertainty(model_cpu, fp_demo.unsqueeze(0), bs_ckpt, cal_factors, device='cpu')
from evaluate import OUTPUT_NAMES
print('示例预测:')
for i, name in enumerate(OUTPUT_NAMES):
    print(f'  {name}: pred={mean_pred[i]:.4f} ± {half_width[i]:.4f} | true={lbl_demo[i]:.4f}')

## 完成后
1. 在 `/root/autodl-tmp/checkpoints/` 确认 `bootstrap_heads.pth` 和 `calibration.json` 已生成
2. 使用 AutoDL 文件管理器或 scp 下载到本地 `checkpoints/`
3. 验证 `calibration.json` 中 `empirical_coverage_after` 所有值 >= 0.95